# Topographic corrections

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

In [ ]:
from evaspa import daily, io, seb, solar

In [ ]:
def plot_dem(xrds_dem: xr.Dataset):
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 4))
    xrds_dem["height"].plot(
        ax=axes[0],
        vmin=xrds_dem["height"].min(),
        vmax=xrds_dem["height"].max(),
        cmap="RdYlGn_r",
    )
    xrds_dem["slope"].plot(
        ax=axes[1],
        vmin=xrds_dem["slope"].min(),
        vmax=xrds_dem["slope"].max(),
        cmap="Reds",
    )
    xrds_dem["aspect"].plot(
        ax=axes[2],
        vmin=xrds_dem["aspect"].min(),
        vmax=xrds_dem["aspect"].max(),
        cmap="twilight_shifted",
    )


def plot(data1: xr.DataArray, data2: xr.DataArray):
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))
    data1.plot(
        ax=axes[0],
        cmap="Reds",
    )
    data2.plot(
        ax=axes[1],
        cmap="Reds",
    )

## Instant direct downward shortwave radiation

Read DEM

In [ ]:
dem = io.read_data_from_file(Path("../tests/data/dem.tif"))

In [ ]:
dem

In [ ]:
plot_dem(dem)

Homogenous downward shortwave radiation

In [ ]:
date = dt.datetime(2025, 1, 12, 13, 00, 00, tzinfo=dt.timezone.utc)

In [ ]:
rsd = xr.Dataset(
    data_vars={
        "rsd": dem["height"].copy(data=np.ones(dem.height.shape) * 500),
        "fdiff": dem["height"].copy(data=np.ones(dem.height.shape) * 0.2),
    },
    attrs=dem.attrs.copy(),
)

In [ ]:
rsd

Compute SZA ad SAA

In [ ]:
sza, saa = solar.compute_sun_angles(
    date=date, x=rsd.coords["x"], y=rsd.coords["y"], crs=rsd.attrs["crs"]
)
rsd["sza"] = rsd["rsd"].copy(data=sza)
rsd["saa"] = rsd["rsd"].copy(data=saa)

Correct direct downward radiation

In [ ]:
rsd["corr"] = seb.correct_shortwave_radiation(
    rsd=rsd["rsd"],
    sza=rsd["sza"],
    saa=rsd["saa"],
    date=date,
    slope=dem["slope"],
    aspect=dem["aspect"],
)

In [ ]:
rsd["corr"].plot(cmap="Reds")

## Instant and daily TOA solar radiation

In [ ]:
date = dt.datetime(2025, 1, 12, 14, 00, 00, tzinfo=dt.timezone.utc)

TOA solar radiation

In [ ]:
toa_inst = solar.compute_toa_solar_radiation(
    date,
    x=rsd.coords["x"],
    y=rsd.coords["y"],
    crs=rsd.attrs["crs"],
    slope=dem["slope"],
    aspect=dem["aspect"],
)
rsd["toa_inst"] = rsd["rsd"].copy(data=toa_inst)

In [ ]:
rsd["toa_inst"].plot(cmap="Reds")

In [ ]:
toa_daily = solar.compute_daily_toa_solar_radiation(
    date,
    x=rsd.coords["x"],
    y=rsd.coords["y"],
    crs=rsd.attrs["crs"],
    slope=dem["slope"],
    aspect=dem["aspect"],
)
rsd["toa_daily"] = rsd["rsd"].copy(data=toa_daily)

In [ ]:
rsd["toa_daily"].plot(cmap="Reds")

In [ ]:
toa_daily = solar.compute_daily_toa_solar_radiation_from_hour_angle(
    date,
    x=rsd.coords["x"],
    y=rsd.coords["y"],
    crs=rsd.attrs["crs"],
    slope=dem["slope"],
    aspect=dem["aspect"],
)
rsd["toa_daily_acc"] = rsd["rsd"].copy(data=toa_daily)

In [ ]:
rsd["toa_daily_acc"].plot(cmap="Reds")

## Correct shortwave radiation

In [ ]:
# Read RSD
data = io.read_data_from_file("../tests/data/modis_test_dem.tif")
rsd = data["rsd"]
dem = data[["aspect", "slope"]]

In [ ]:
# Correct
rsd_corr = seb.correct_shortwave_radiation(
    rsd=rsd, date=date, crs=data.crs, slope=dem.slope, aspect=dem.aspect
)

In [ ]:
plot(rsd, rsd_corr)

## Extrapolate at daily scale

In [ ]:
# Read RSD
data = io.read_data_from_file("../tests/data/modis_test_dem.tif")
inst_rsd = data[["rsd"]]
dem = data[["aspect", "slope"]]
inst_rsd["rsd_corr"] = seb.correct_shortwave_radiation(
    rsd=inst_rsd["rsd"],
    date=date,
    crs=data.crs,
    slope=dem.slope,
    aspect=dem.aspect,
)
inst_rsd.attrs["date"] = date

In [ ]:
plot(inst_rsd["rsd"], inst_rsd["rsd_corr"])

In [ ]:
# Extrapolate with DEM at daily
daily_rsd = daily.extrapolate_at_daily_scale(
    data=inst_rsd, dem=dem, method="toa", use_topo=True
)

In [ ]:
plot(daily_rsd["rsd"], daily_rsd["rsd_corr"])